# 08 — RAG fra bunnen av

**Fase:** 2 — Kjerne AI | **Tid:** 2–3 timer | **Krav:** Notatbok 05, 06, 07

**Hva du bygger:** Et fungerende RAG-system (Retrieval-Augmented Generation) som svarer på pensjonsspørsmål basert på egne dokumenter — uten noe rammeverk, bare ren Python.

---

## Hva er RAG?

Et LLM alene kan bare svare basert på det det lærte under trening. Det kjenner ikke dine interne dokumenter, nyeste regelverk eller bedriftsdata.

**RAG løser dette i to steg:**

```
1. RETRIEVE  → Finn de mest relevante tekstbitene for spørsmålet
2. GENERATE  → Send spørsmål + biter til LLM og få svar
```

Full pipeline:
```
Dokument → Chunk → Embed → Lagre i vektordatabase
                                    ↓
Spørsmål → Embed → Søk → Hent topp-K → Bygg prompt → LLM → Svar
```

---

## Oppsett: Gratis verktøy

- **Ollama** — gratis lokal LLM (se notatbok 05)
- **sentence-transformers** — gratis embeddings
- **ChromaDB** — gratis vektordatabase

In [ ]:
%pip install -q chromadb sentence-transformers openai

In [ ]:
import chromadb
from sentence_transformers import SentenceTransformer
from openai import OpenAI

# Gratis embedding-modell
embed_modell = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

# Gratis lokal LLM via Ollama
llm = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")
LLM_MODELL = "llama3.2"  # Bytt til modellen du har installert

# Vektordatabase
db = chromadb.PersistentClient(path="./rag_db")
samling = db.get_or_create_collection("spk_docs", metadata={"hnsw:space": "cosine"})

print("Alle komponenter klare.")

---

## Steg 1: Ingest — Last inn og chunk dokumenter

In [ ]:
# Simulerte SPK-dokumenter (i praksis: PDF, Word, nettside, database)
DOKUMENTER = [
    {
        "id": "spk-afp",
        "tittel": "AFP-guide",
        "tekst": """AFP (avtalefestet pensjon) er en pensjonsordning for offentlig ansatte.
Du kan ta ut AFP fra du er 62 år gammel.
For å ha rett til AFP må du ha jobbet i offentlig sektor i minst tre år.
AFP utbetales livsvarig og kan kombineres med alderspensjon.
Størrelsen på AFP avhenger av lønn og antall år du har jobbet.
Du søker om AFP via din arbeidsgiver eller direkte til SPK.
Søknadsfristen er tre måneder før ønsket startdato."""
    },
    {
        "id": "spk-alderspensjon",
        "tittel": "Alderspensjon",
        "tekst": """Alderspensjon fra SPK utbetales livsvarig fra fylte 67 år.
Du kan velge å ta ut pensjonen delvis fra du er 62 år.
Full pensjon krever 30 år i pensjonsordningen.
Pensjonen beregnes som en prosent av lønnen din ved pensjonering.
Samlet pensjon fra SPK og NAV kan ikke overstige 70 prosent av sluttlønnen."""
    },
    {
        "id": "spk-uføre",
        "tittel": "Uførepensjon",
        "tekst": """Uførepensjon gis ved varig nedsatt arbeidsevne på minst 20 prosent.
Du søker via NAV, men SPK utbetaler tillegget for statsansatte.
Satsen er 66 prosent av pensjonsgivende inntekt.
Uførepensjonen løper til du fyller 67 år, da konverteres den til alderspensjon."""
    },
]

def chunk_setningsvis(tekst: str, maks_ord: int = 40) -> list[str]:
    """Del i setninger, men slå sammen veldig korte."""
    setninger = [s.strip() for s in tekst.split(".") if s.strip()]
    chunks, gjeldende = [], ""
    for s in setninger:
        if len((gjeldende + s).split()) > maks_ord and gjeldende:
            chunks.append(gjeldende.strip())
            gjeldende = s + ". "
        else:
            gjeldende += s + ". "
    if gjeldende:
        chunks.append(gjeldende.strip())
    return chunks

# Ingest alle dokumenter
alle_chunks, alle_ids, alle_meta = [], [], []

for dok in DOKUMENTER:
    chunks = chunk_setningsvis(dok["tekst"])
    for i, chunk in enumerate(chunks):
        alle_chunks.append(chunk)
        alle_ids.append(f"{dok['id']}-chunk-{i}")
        alle_meta.append({"kilde": dok["id"], "tittel": dok["tittel"]})

print(f"{len(DOKUMENTER)} dokumenter → {len(alle_chunks)} chunks")

## Steg 2: Embed og lagre

In [ ]:
vektorer = embed_modell.encode(alle_chunks).tolist()

samling.add(
    documents=alle_chunks,
    embeddings=vektorer,
    ids=alle_ids,
    metadatas=alle_meta,
)

print(f"{samling.count()} chunks lagret i vektordatabasen.")
print("\nEksempel-chunks:")
for chunk in alle_chunks[:3]:
    print(f"  → {chunk}")

## Steg 3: Retrieve — Hent relevante chunks

In [ ]:
def hent_kontekst(spørsmål: str, topp_k: int = 3) -> tuple[str, list]:
    spørsmål_vektor = embed_modell.encode([spørsmål]).tolist()
    
    resultater = samling.query(
        query_embeddings=spørsmål_vektor,
        n_results=topp_k,
    )
    
    chunks = resultater["documents"][0]
    meta   = resultater["metadatas"][0]
    
    # Bygg konteksttekst
    kontekst = "\n\n".join(
        f"[{m['tittel']}]\n{c}"
        for c, m in zip(chunks, meta)
    )
    return kontekst, meta

spørsmål = "Kan jeg pensjonere meg ved 62 år?"
kontekst, kilder = hent_kontekst(spørsmål)

print(f"Spørsmål: {spørsmål}")
print(f"\nHentet kontekst:\n{'-'*40}")
print(kontekst)

## Steg 4: Generate — Send til LLM og få svar

In [ ]:
def bygg_prompt(spørsmål: str, kontekst: str) -> list:
    return [
        {
            "role": "system",
            "content": """Du er en pensjonsrådgiver hos SPK.
Svar KUN basert på konteksten nedenfor.
Hvis svaret ikke finnes i konteksten, si: 'Jeg fant ikke den informasjonen i dokumentene.'
Svar på norsk, kortfattet og presist."""
        },
        {
            "role": "user",
            "content": f"""Kontekst:
{kontekst}

Spørsmål: {spørsmål}"""
        }
    ]

def rag_svar(spørsmål: str) -> str:
    kontekst, kilder = hent_kontekst(spørsmål)
    prompt = bygg_prompt(spørsmål, kontekst)
    
    svar = llm.chat.completions.create(
        model=LLM_MODELL,
        messages=prompt,
        temperature=0.1,
    )
    
    tekst = svar.choices[0].message.content
    kilde_titler = list({m['tittel'] for m in kilder})
    return tekst, kilde_titler

# Test systemet
for q in [
    "Kan jeg pensjonere meg ved 62 år?",
    "Hva er satsen for uførepensjon?",
    "Hva koster en flyreise til Tromsø?",  # Skal si at den ikke vet
]:
    svar, kilder = rag_svar(q)
    print(f"Q: {q}")
    print(f"A: {svar}")
    print(f"Kilder: {kilder}")
    print()

---

## Fullstendig RAG-klasse

In [ ]:
class RAGSystem:
    """Enkel RAG-pipeline: ingest, retrieve, generate."""
    
    def __init__(self, embed_modell, llm_klient, llm_modell, db_sti="./rag_system"):
        self.embed = embed_modell
        self.llm   = llm_klient
        self.model = llm_modell
        
        db = chromadb.PersistentClient(path=db_sti)
        self.samling = db.get_or_create_collection(
            "dokumenter", metadata={"hnsw:space": "cosine"}
        )
    
    def legg_til(self, tekst: str, dok_id: str, metadata: dict = None):
        """Chunk, embed og lagre ett dokument."""
        chunks = chunk_setningsvis(tekst)
        vektorer = self.embed.encode(chunks).tolist()
        self.samling.add(
            documents=chunks,
            embeddings=vektorer,
            ids=[f"{dok_id}-{i}" for i in range(len(chunks))],
            metadatas=[metadata or {} for _ in chunks],
        )
    
    def spør(self, spørsmål: str, topp_k: int = 3) -> dict:
        """Returner svar og kildetekster."""
        sv = self.embed.encode([spørsmål]).tolist()
        res = self.samling.query(query_embeddings=sv, n_results=topp_k)
        kontekst = "\n\n".join(res["documents"][0])
        
        svar = self.llm.chat.completions.create(
            model=self.model,
            messages=[
                {"role": "system",  "content": "Svar på norsk basert på konteksten. Si fra om du ikke finner svaret."},
                {"role": "user",    "content": f"Kontekst:\n{kontekst}\n\nSpørsmål: {spørsmål}"},
            ],
            temperature=0.1,
        )
        return {
            "svar": svar.choices[0].message.content,
            "kontekst": res["documents"][0],
        }

print("RAGSystem-klasse klar!")

---

## Oppsummering

Du har bygget en komplett RAG-pipeline:

```
Dokumenter → chunk_setningsvis() → embed() → ChromaDB
Spørsmål   → embed() → samling.query() → bygg_prompt() → LLM → Svar
```

**Vanlige problemer:**
- Dårlig chunking → relevant innhold deles på tvers av chunks → dårlige treff
- For mange/få chunks hentet (topp_k) → for mye støy / misser svaret
- LLM hallusinerer utenfor konteksten → løses med strenge system-prompts

---

## Hva er neste steg?

**Neste: `09_rag_advanced.ipynb`** — Gjør RAG bedre med hybrid søk (BM25 + semantisk), reranking og evaluering med RAGAS.